# Evaluating joint acquisition: re-analysis of released results

**Abdullah Hamza, Independent Researcher.** This notebook accompanies *Evaluating fixed-kernel joint acquisition for cross-context perturbation prediction*, a technical report / public research release. Hosting is not peer review or journal acceptance. No DOI is assigned here.

We ask whether a fixed source-coordinate RBF variance term improves a greedy worst-query acquisition rule when both feed the same paid-validation ridge predictor. Calibration identities, cell depth, and shared controls share a hypothetical recipient budget. Query interventions have measured source responses but sealed recipient outcomes.

This is an **executable re-analysis of saved outputs**, not new biological data, a rerun of raw-data preparation, or a repetition of the original acquisition/training/scoring experiments. It recomputes the primary budget-normalized areas and all 32 later robustness contrasts, checks them against the retained results, and creates two figures and CSV/JSON outputs. It requires only the attached compact input tables, Python and Matplotlib; no internet, GPU, credentials or package installation.

[GitHub project](https://github.com/abdullah2hamza-blip/MLResearch) · [Full reproduction instructions](https://github.com/abdullah2hamza-blip/MLResearch/blob/main/docs/external-reproduction.md) · [Pinned upstream data](https://zenodo.org/records/13350497)


## Method, provenance and interpretation

The experimental target is a difference of recipient **cell-level normalized-log expression means**, using the full released panel for each library normalization before source/recipient gene intersection. The original K562/RPE1 replay uses 131 query interventions, 7,226 aligned genes, five acquisition-order/tie seeds and budgets 128/256/512. Those seeds do not create new biological replicates. The later post-primary matrix crosses two new split/cell-order seeds (20261023/20261024), ranks 4/16, two directions and budgets through 2,048, using three acquisition seeds. The repeated tasks reuse observations.

Worst-query and mean-query optimize working variance criteria. The fixed RBF term has unit amplitude and is not a fitted discrepancy component of the ridge predictor. Its surrogate variance is not the realized predictor's risk. A numerical source-coordinate span certificate does not establish biological transportability.

The input manifest identifies the public release commit/tag, upstream scPerturb record 13350497 version 1.4, scientific software versions and exact table checksums. Original primary results use their frozen v0.1.0 source; robustness uses the retained v0.2.0 numerical source. A later correction to a transfer-acceptance diagnostic did not change the original prediction arrays or scores; consult the manuscript and reproduction guide for strict historical replay.

Raw expression, full prediction arrays and model weights are absent from this compact input. Obtain pinned raw inputs using the full project's acquisition instructions if you want the larger experimental reproduction.


In [ ]:
import csv
import hashlib
import importlib.metadata
import json
import math
import os
import platform
import statistics
import sys
import time
from collections import defaultdict
from pathlib import Path, PurePosixPath

import matplotlib
import matplotlib.pyplot as plt

started = time.perf_counter()
manifest_name = "mlresearch-results-manifest.json"
explicit_root = os.environ.get("MLRESEARCH_INPUT_ROOT")
if explicit_root:
    candidates = [Path(explicit_root).resolve() / manifest_name]
else:
    mounted_inputs = Path("/kaggle/input")
    candidates = sorted(mounted_inputs.rglob(manifest_name)) if mounted_inputs.is_dir() else []
if len(candidates) != 1 or not candidates[0].is_file():
    raise ValueError(
        "Attach exactly one MLResearch result-input release, or set "
        "MLRESEARCH_INPUT_ROOT to its directory."
    )
manifest_path = candidates[0]
input_root = manifest_path.parent.resolve()
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
required_inputs = {"primary_curves", "paired_query_areas", "paired_area_comparisons"}
if manifest.get("schema_version") != 1 or set(manifest.get("files", {})) != required_inputs:
    raise ValueError("Unsupported or incomplete result-input manifest")
release = manifest.get("release", {})
if not release.get("commit") or not release.get("tag"):
    raise ValueError("The input release must identify its source commit and tag")
paths = {}
input_hashes = {}
for name, item in manifest["files"].items():
    relative = PurePosixPath(item["path"])
    if relative.is_absolute() or ".." in relative.parts or ":" in item["path"]:
        raise ValueError("Unsafe input-table path")
    path = (input_root / relative).resolve()
    if not path.is_relative_to(input_root):
        raise ValueError("An input resolves outside its release directory")
    digest = hashlib.sha256(path.read_bytes()).hexdigest()
    if digest != item["sha256"]:
        raise ValueError("Changed input table: " + name)
    paths[name] = path
    input_hashes[name] = digest
output_root = Path(
    os.environ.get("MLRESEARCH_OUTPUT_ROOT", "/kaggle/working/mlresearch-reanalysis")
).resolve()
output_root.mkdir(parents=True, exist_ok=True)
(output_root / "reanalysis-verification.json").write_text(
    json.dumps({"state": "running"}) + "\n", encoding="utf-8"
)
print("Release:", json.dumps(release, sort_keys=True))
print("Upstream:", json.dumps(manifest.get("upstream_data", {}), sort_keys=True))
print("Verified input tables:", len(paths))
print("Python:", sys.version.split()[0], "Matplotlib:", matplotlib.__version__)

## Primary comparison from the frozen saved curves

For budget checkpoints $b_1<\cdots<b_J$, the endpoint is the trapezoid integral of full-gene/query/seed mean squared error divided by $b_J-b_1$. There are no observations below the first budget. Percent changes divide the extension-minus-reference area by **that reference's own area**; positive means larger extension error.

The plotted zero line is a standalone zero-recipient-measurement-use reference. Its horizontal display does not refund another policy's acquisitions. The scorer-noise line is a plug-in independent-cell scale estimate, not known population truth or an irreducible biological bound. Curves below are descriptive averages, with no uncertainty bars.


In [ ]:
def read_rows(path):
    with path.open(encoding="utf-8", newline="") as stream:
        return list(csv.DictReader(stream))


def finite(value):
    result = float(value)
    if not math.isfinite(result):
        raise ValueError("Non-finite numeric result")
    return result


def normalized_area(rows, field="mse"):
    ordered = sorted(rows, key=lambda row: int(row["budget"]))
    budgets = [int(row["budget"]) for row in ordered]
    if budgets != [128, 256, 512]:
        raise ValueError("Primary curve must have exactly the 128/256/512 checkpoints")
    values = [finite(row[field]) for row in ordered]
    return math.fsum(
        (b - a) * (x + y) / 2
        for a, b, x, y in zip(budgets[:-1], budgets[1:], values[:-1], values[1:], strict=True)
    ) / (budgets[-1] - budgets[0])


primary_rows = read_rows(paths["primary_curves"])
runs = ["external-k562-to-rpe1-v1", "external-rpe1-to-k562-v1"]
policies = ["bias_aware", "worst_query", "mean_query", "reliability"]
curves = defaultdict(list)
for row in primary_rows:
    if row["run"] in runs and row["source_control"] == "none":
        if row["backbone"] == "ridge_cv" and row["policy"] in policies:
            curves[(row["run"], row["policy"])].append(row)
        elif row["backbone"] == "zero" and row["policy"] == "bias_aware":
            curves[(row["run"], "zero")].append(row)
areas = {}
primary_areas = []
primary_comparisons = []
for run in runs:
    for policy in [*policies, "zero"]:
        rows = curves[(run, policy)]
        if any(int(row["query_seed_predictions"]) != 131 * 5 for row in rows):
            raise ValueError("Unexpected primary query/seed denominator")
        area = normalized_area(rows)
        areas[(run, policy)] = area
        primary_areas.append(
            {
                "run": run,
                "policy": policy,
                "backbone": "zero" if policy == "zero" else "ridge_cv",
                "budget_min": 128,
                "budget_max": 512,
                "mse_area": area,
            }
        )
    extension = areas[(run, "bias_aware")]
    for reference in policies[1:]:
        baseline = areas[(run, reference)]
        if baseline <= 0:
            raise ValueError("Nonpositive reference area")
        primary_comparisons.append(
            {
                "run": run,
                "reference": reference,
                "extension_area": extension,
                "reference_area": baseline,
                "paired_difference": extension - baseline,
                "relative_change_percent": 100 * (extension - baseline) / baseline,
            }
        )
    zero_gain = 100 * (areas[(run, "zero")] - extension) / areas[(run, "zero")]
    print(
        run,
        "extension area:",
        format(extension, ".12g"),
        "reduction versus zero:",
        format(zero_gain, ".6f") + "%",
    )
for row in primary_comparisons:
    print(
        row["run"], "versus", row["reference"], format(row["relative_change_percent"], "+.7f") + "%"
    )
expected_primary_percent = {runs[0]: 0.5650687, runs[1]: 0.0137794}
for row in primary_comparisons:
    if row["reference"] == "worst_query" and not math.isclose(
        row["relative_change_percent"],
        expected_primary_percent[row["run"]],
        abs_tol=0.0000001,
        rel_tol=0,
    ):
        raise ValueError("Primary result differs from the manuscript's reported precision")

In [ ]:
labels = {
    "bias_aware": "Fixed-RBF extension",
    "worst_query": "Worst-query",
    "mean_query": "Mean-query",
    "reliability": "Reliability heuristic",
    "zero": "Standalone zero",
}
colors = {
    "bias_aware": "#7b3294",
    "worst_query": "#d95f02",
    "mean_query": "#1f78b4",
    "reliability": "#238b45",
    "zero": "#666666",
}
fig, axes = plt.subplots(1, 2, figsize=(12, 4.7), sharex=True)
for ax, run, title in zip(axes, runs, ["K562 to RPE1", "RPE1 to K562"], strict=True):
    maximum = 0
    for policy in [*policies, "zero"]:
        rows = sorted(curves[(run, policy)], key=lambda row: int(row["budget"]))
        budgets = [int(row["budget"]) for row in rows]
        values = [finite(row["mse"]) * 1000 for row in rows]
        maximum = max(maximum, *values)
        ax.plot(
            budgets,
            values,
            marker="o",
            color=colors[policy],
            linestyle="--" if policy == "zero" else "-",
            label=labels[policy],
        )
    noise = normalized_area(curves[(run, "bias_aware")], "assessment_noise_floor") * 1000
    ax.axhline(noise, color="#999999", linestyle=":", label="Plug-in scorer noise")
    ax.set(
        title=title,
        xlabel="Recipient budget ceiling (abstract units)",
        ylabel="Normalized-log MSE (×10⁻³)",
        ylim=(0, maximum * 1.08),
        xticks=[128, 256, 512],
    )
    ax.grid(alpha=0.22)
handles, legend_labels = axes[0].get_legend_handles_labels()
fig.legend(handles, legend_labels, loc="lower center", ncol=3, frameon=False)
fig.suptitle("Original frozen replay: shared paid-validation ridge predictor")
fig.tight_layout(rect=(0, 0.15, 1, 0.94))
fig.savefig(output_root / "primary-curves.png", dpi=160, bbox_inches="tight")
plt.show()
plt.close(fig)

## Every later robustness contrast

The post-primary study has eight direction/split/rank cells. Each is compared with both worst-query and mean-query on the original (128–512) and full (128–2,048) budget grids: **32 contrasts in total**. Below we recompute each contrast from the 131 paired query areas, not merely copy its reported summary.

The query areas already average the three acquisition seeds. The retained conditional query-resampling ranges in the summary CSV are retained metadata; this notebook does **not** regenerate those 2,000 bootstrap draws, cell resampling, technical-batch sensitivity, or original fits. They are conditional on the observed queries/source/fits/scoring banks, not biological confidence intervals. Colors in the next figure show descriptive point changes; no multiplicity-adjusted winning subgroup is inferred.


In [ ]:
paired = read_rows(paths["paired_query_areas"])
retained = read_rows(paths["paired_area_comparisons"])
key_fields = ("run", "grid", "reference")
groups = defaultdict(list)
for row in paired:
    groups[tuple(row[key] for key in key_fields)].append(row)
retained_by_key = {}
for row in retained:
    key = tuple(row[field] for field in key_fields)
    if key in retained_by_key:
        raise ValueError("Duplicate retained robustness contrast")
    retained_by_key[key] = row
if len(groups) != 32 or set(groups) != set(retained_by_key):
    raise ValueError("Incomplete robustness comparison matrix")
robustness = []
max_mean_discrepancy = 0
max_relative_discrepancy = 0
for key, rows in sorted(groups.items()):
    original = retained_by_key[key]
    if len(rows) != 131 or len({row["query"] for row in rows}) != 131:
        raise ValueError("Each robustness contrast must contain 131 distinct query identities")
    for field in ("split_seed", "source_context", "target_context", "rank"):
        if len({row[field] for row in rows}) != 1 or rows[0][field] != original[field]:
            raise ValueError("Inconsistent robustness metadata")
    extension = [finite(row["extension_area"]) for row in rows]
    reference = [finite(row["reference_area"]) for row in rows]
    differences = [a - b for a, b in zip(extension, reference, strict=True)]
    for row, difference in zip(rows, differences, strict=True):
        if not math.isclose(difference, finite(row["paired_difference"]), abs_tol=1e-14, rel_tol=0):
            raise ValueError("A paired query difference does not match its two areas")
    extension_area = statistics.fmean(extension)
    reference_area = statistics.fmean(reference)
    if reference_area <= 0:
        raise ValueError("Nonpositive robustness reference area")
    difference = statistics.fmean(differences)
    relative = difference / reference_area
    mean_error = abs(difference - finite(original["paired_difference"]))
    relative_error = abs(relative - finite(original["relative_change"]))
    max_mean_discrepancy = max(max_mean_discrepancy, mean_error)
    max_relative_discrepancy = max(max_relative_discrepancy, relative_error)
    if mean_error > 1e-14 or relative_error > 1e-12:
        raise ValueError("Recomputed robustness result disagrees with its retained summary")
    if not math.isclose(
        extension_area, finite(original["extension_area"]), abs_tol=1e-14, rel_tol=0
    ) or not math.isclose(
        reference_area, finite(original["reference_area"]), abs_tol=1e-14, rel_tol=0
    ):
        raise ValueError("Robustness absolute areas disagree with their retained summary")
    total = math.fsum(differences)
    loo = [(total - value) / (len(rows) - 1) for value in differences]
    robustness.append(
        {
            **{
                field: original[field]
                for field in (
                    "run",
                    "split_seed",
                    "source_context",
                    "target_context",
                    "rank",
                    "grid",
                    "reference",
                )
            },
            "queries": len(rows),
            "extension_area": extension_area,
            "reference_area": reference_area,
            "paired_difference": difference,
            "relative_change_percent": 100 * relative,
            "queries_with_lower_extension_error": sum(value < 0 for value in differences),
            "query_difference_median": statistics.median(differences),
            "leave_one_query_out_mean_min": min(loo),
            "leave_one_query_out_mean_max": max(loo),
            "retained_conditional_query_low": finite(original["conditional_query_low"]),
            "retained_conditional_query_high": finite(original["conditional_query_high"]),
        }
    )
counts = {}
for grid in ("original_grid", "full_grid"):
    for reference in ("worst_query", "mean_query"):
        selected = [
            row for row in robustness if row["grid"] == grid and row["reference"] == reference
        ]
        if len(selected) != 8:
            raise ValueError("Missing direction/split/rank cell")
        count = sum(row["paired_difference"] < 0 for row in selected)
        counts[grid + "/" + reference] = count
        print(
            grid,
            "versus",
            reference + ":",
            count,
            "of",
            len(selected),
            "extension point estimates have lower error",
        )
expected_counts = {
    "original_grid/worst_query": 3,
    "original_grid/mean_query": 2,
    "full_grid/worst_query": 2,
    "full_grid/mean_query": 5,
}
if counts != expected_counts:
    raise ValueError("Robustness sign counts differ from the manuscript")
favorable = min(
    (row for row in robustness if row["grid"] == "full_grid" and row["reference"] == "worst_query"),
    key=lambda row: row["paired_difference"],
)
if favorable["queries_with_lower_extension_error"] != 47 or not math.isclose(
    favorable["query_difference_median"], 0.00013621101886853847, abs_tol=1e-14, rel_tol=0
):
    raise ValueError(
        "Favorable-cell query distribution differs from the retained manuscript result"
    )
print("Descriptively most favorable full-grid cell versus worst-query:", favorable["run"])
print(
    "Mean percent change:",
    round(favorable["relative_change_percent"], 6),
    "; queries with lower extension error:",
    favorable["queries_with_lower_extension_error"],
    "/",
    favorable["queries"],
)
print(
    "Query median:",
    favorable["query_difference_median"],
    "; leave-one-query-out mean range:",
    [favorable["leave_one_query_out_mean_min"], favorable["leave_one_query_out_mean_max"]],
)

In [ ]:
cell_keys = sorted(
    {
        (row["source_context"], row["target_context"], int(row["split_seed"]), int(row["rank"]))
        for row in robustness
    }
)
lookup = {
    (
        row["source_context"],
        row["target_context"],
        int(row["split_seed"]),
        int(row["rank"]),
        row["grid"],
        row["reference"],
    ): row
    for row in robustness
}
limit = max(abs(row["relative_change_percent"]) for row in robustness)
fig, axes = plt.subplots(1, 2, figsize=(12, 6.6), sharey=True)
for ax, grid, title in zip(
    axes, ["original_grid", "full_grid"], ["Budgets 128–512", "Budgets 128–2,048"], strict=True
):
    matrix = [
        [
            lookup[(*key, grid, reference)]["relative_change_percent"]
            for reference in ["worst_query", "mean_query"]
        ]
        for key in cell_keys
    ]
    image = ax.imshow(matrix, cmap="RdBu_r", vmin=-limit, vmax=limit, aspect="auto")
    for i, values in enumerate(matrix):
        for j, value in enumerate(values):
            ax.text(
                j,
                i,
                f"{value:+.3f}%",
                ha="center",
                va="center",
                color="white" if abs(value) > limit * 0.65 else "black",
                fontsize=10,
            )
    ax.set(title=title, xticks=[0, 1], xticklabels=["vs worst-query", "vs mean-query"])
    ax.set_yticks(
        range(len(cell_keys)),
        [
            f"{source}→{target} | split {str(seed)[-2:]} | rank {rank}"
            for source, target, seed, rank in cell_keys
        ],
    )
fig.subplots_adjust(left=0.30, right=0.86, wspace=0.15, top=0.89, bottom=0.09)
color_ax = fig.add_axes((0.89, 0.14, 0.018, 0.68))
fig.colorbar(image, cax=color_ax, label="Extension/reference area change (%)")
fig.suptitle("All post-primary robustness contrasts: negative favors the extension")
fig.savefig(output_root / "robustness-matrix.png", dpi=160, bbox_inches="tight")
plt.show()
plt.close(fig)

## What the results do and do not show

The original primary comparisons give no demonstrated benefit over worst-query; the point changes are approximately +0.565% and +0.0138% error. These are not proofs of equivalence or absence of benefit. In the wider post-primary matrix, six of eight point contrasts favor worst-query, while five favor the extension against mean-query. One forward rank-four cell improves by about 3.58% against worst-query, but only 47/131 query differences improve and the median query difference worsens. Deleting any single query leaves its aggregate mean benefit negative. This conditional success already appears on that split's shorter grid, so extending the budget alone cannot be identified as its cause.

The directions, splits and budgets share cells and queries. Their contrasts are not eight independent biological trials. A data-selected favorable cell is descriptive, with no multiplicity-adjusted biological inference. Other manuscript limitations include a count-selected target cohort, protocol/effector/time differences, noisy assessment contrasts, estimated source coordinates, strong shrinkage, a reconstructed neural comparator and hypothetical action prices. The fixed-kernel surrogate's mixed behavior does not reject discrepancy-aware acquisition in general.

This notebook verifies arithmetic and selected retained evidence. It does not independently establish original raw-data provenance, verify every prediction array, repeat all paper diagnostics, infer new biology, validate adaptive confidence intervals or establish prospective laboratory savings. The full project supplies the larger data acquisition, freeze, scoring and reproduction paths.


In [ ]:
def write_csv(path, rows):
    if not rows:
        raise ValueError("No output rows")
    with path.open("w", encoding="utf-8", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)


write_csv(output_root / "primary-areas.csv", primary_areas)
write_csv(output_root / "primary-comparisons.csv", primary_comparisons)
write_csv(output_root / "robustness-contrasts.csv", robustness)
outputs = [
    "primary-areas.csv",
    "primary-comparisons.csv",
    "robustness-contrasts.csv",
    "primary-curves.png",
    "robustness-matrix.png",
]
verification = {
    "state": "passed_saved_result_reanalysis",
    "release": release,
    "input_manifest_sha256": hashlib.sha256(manifest_path.read_bytes()).hexdigest(),
    "intended_notebook_source_sha256_from_manifest": manifest.get("notebook_source_sha256"),
    "input_sha256": input_hashes,
    "primary_directions": len(runs),
    "primary_budget_checkpoints": [128, 256, 512],
    "robustness_contrasts_recomputed": len(robustness),
    "paired_query_rows_checked": len(paired),
    "queries_per_robustness_contrast": 131,
    "maximum_absolute_paired_mean_discrepancy": max_mean_discrepancy,
    "maximum_absolute_relative_change_discrepancy": max_relative_discrepancy,
    "robustness_extension_lower_error_counts": counts,
    "conditional_ranges": "Retained source ranges, not regenerated bootstrap draws",
    "output_sha256": {
        name: hashlib.sha256((output_root / name).read_bytes()).hexdigest() for name in outputs
    },
    "runtime": {
        "python": sys.version.split()[0],
        "matplotlib": matplotlib.__version__,
        "numpy": importlib.metadata.version("numpy"),
        "platform": platform.platform(),
        "cpu_count": os.cpu_count(),
        "elapsed_seconds": time.perf_counter() - started,
    },
    "verification_category": "Re-analysis of saved outputs; not full experimental reproduction",
}
(output_root / "reanalysis-verification.json").write_text(
    json.dumps(verification, indent=2, sort_keys=True, allow_nan=False) + "\n", encoding="utf-8"
)
print("Saved outputs:", ", ".join([*outputs, "reanalysis-verification.json"]))
print(
    "Completed saved-result reanalysis:",
    len(robustness),
    "contrasts; maximum mean discrepancy",
    max_mean_discrepancy,
)

## Citation, licenses and AI assistance

Cite Abdullah Hamza, *Evaluating fixed-kernel joint acquisition for cross-context perturbation prediction*, and identify the release commit/tag printed above. Use the repository's `CITATION.cff` for software metadata. Cite the scPerturb and original experimental sources described in the manuscript; no new DOI or peer-review status is invented here.

Notebook code is original project software under MIT. Narrative derived from the manuscript retains its CC BY 4.0 attribution. The attached input is a **mixed-artifact container**: preserve the per-file scope and original notices in `licenses/SCOPE.txt`, `SOFTWARE-LICENSE.txt` and `MANUSCRIPT-LICENSE.txt`. Original retained CSVs and upstream materials keep their established terms; a Kaggle container label does not relicense them.

**AI assistance.** OpenAI Codex (OpenAI) assisted with research planning, literature review and experimental design, code preparation, statistical analysis and interpretation, figure and table preparation, manuscript drafting and revision, internal review, and release preparation. These AI-assisted reviews were internal quality checks, not independent human peer review. The human author retains responsibility for the final work.
